# Hugging Face MCP + Notion MCP: AI 최신 논문 → 모델 허브 검색 → Notion 정리

- **[Hugging Face MCP](https://huggingface.co/mcp)**  
  Hub에서 논문(papers), 모델(models), 데이터셋(datasets) 검색

- **Notion MCP**  
  Notion 페이지 생성·수정 (반드시 Notion MCP 도구 사용)

- **MCP 우선, LLM 보조**  
  논문·모델·데이터셋 정보는 Hugging Face MCP(paper_search, hub_repo_search, hub_repo_details)로 수집하고, Notion MCP로 작성.  
  LLM은 구조·포맷(제목, 불릿, 클릭 가능 링크)만 잡고, 설명은 MCP 도구 결과를 그대로 사용.

- **Notion 본문**  
  논문/모델 설명은 HF MCP가 반환한 텍스트를 활용. 모든 URL은 [텍스트](URL) 형태로 넣어 클릭 시 해당 페이지로 이동.

- **매일 1회 실행**  
  이 노트북을 매일 1회 실행(예: cron, GitHub Actions)하면 당일 AI 최신 논문 1편이 부모 페이지 아래 날짜별로 추가됩니다.

## 환경 변수

- **`HF_TOKEN`**  
  [Hugging Face Access Token](https://huggingface.co/settings/tokens) (read 권한 이상). HF MCP 인증용.

- **`NOTION_TOKEN`**  
  Notion 연동 시크릿 (ntn_...). [Notion 연동](https://www.notion.so/profile/integrations)에서 발급.

- **`NOTION_PARENT_PAGE_ID`**  
  Notion에 정리 페이지를 생성할 부모 페이지 ID.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## MCP 클라이언트 설정 (Hugging Face + Notion)

- **Hugging Face**  
  `https://huggingface.co/mcp` (Streamable HTTP), `Authorization: Bearer <HF_TOKEN>`

- **Notion**  
  STDIO, `NOTION_TOKEN` (Notion MCP 필수 사용)

In [2]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient

hf_token = os.getenv("HF_TOKEN", "").strip()
notion_token = os.getenv("NOTION_TOKEN", "").strip()

servers = {}

if hf_token:
    servers["huggingface"] = {
        "transport": "streamable_http",
        "url": "https://huggingface.co/mcp",
        "headers": {"Authorization": f"Bearer {hf_token}"},
    }
else:
    print("[경고] .env에 HF_TOKEN이 없어 Hugging Face MCP가 로드되지 않습니다. https://huggingface.co/settings/tokens 에서 토큰을 발급해 HF_TOKEN=hf_... 로 설정하세요.")

if notion_token:
    servers["notion"] = {
        "command": "npx",
        "args": ["-y", "@notionhq/notion-mcp-server"],
        "env": {"NOTION_TOKEN": notion_token},
        "transport": "stdio",
    }
else:
    print("[경고] .env에 NOTION_TOKEN이 없어 Notion MCP가 로드되지 않습니다.")

assert len(servers) >= 1, "MCP 서버는 최소 1개 이상 필요합니다 (HF_TOKEN 또는 NOTION_TOKEN 설정)."
mcp_client = MultiServerMCPClient(servers)

In [3]:
import time

start = time.perf_counter()
tool_list = await mcp_client.get_tools()
elapsed = time.perf_counter() - start

print(f"get_tools() 소요: {elapsed:.2f}초, 툴 수: {len(tool_list)}")
# 에이전트에는 Notion API-post-page 등 블록 직접 호출 도구 제외 → Notion 생성은 아래 코드에서 create-pages로 직접 호출
agent_tools = [t for t in tool_list if not (t.name.startswith("API-") or "block" in t.name.lower() or "retrieve" in t.name.lower())]
if tool_list:
    print("툴 이름 예시:", [t.name for t in tool_list[:15]])
print("에이전트에 전달한 툴 수:", len(agent_tools))

get_tools() 소요: 1.71초, 툴 수: 30
툴 이름 예시: ['hf_whoami', 'space_search', 'hub_repo_search', 'paper_search', 'hub_repo_details', 'hf_doc_search', 'hf_doc_fetch', 'gr1_z_image_turbo_generate', 'API-get-user', 'API-get-users', 'API-get-self', 'API-post-search', 'API-get-block-children', 'API-patch-block-children', 'API-retrieve-a-block']
에이전트에 전달한 툴 수: 8


## 에이전트 및 스트림 처리

In [4]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model="gpt-4o")
agent = create_react_agent(
    model=llm,
    tools=agent_tools,
    state_modifier="MCP를 최대한 활용하고 LLM은 보조로만 써. (1) 논문·모델·데이터셋 내용은 MCP 도구 결과만 사용. (2) LLM 역할: 본문 구조(###, 표)와 [텍스트](URL) 링크 포맷으로 정리. 논문·모델 설명은 **한 줄 요약 말고 최소 5줄 이상**, **쉬운 말**로 풀어서 써줘. (3) Notion 페이지 생성은 하지 마라. 정리한 본문만 출력.",
)

async def process_stream(stream_generator):
    results = []
    try:
        async for chunk in stream_generator:
            key = list(chunk.keys())[0]
            if key == "agent":
                msg = chunk["agent"]["messages"][0]
                content = msg.content if msg.content else msg.additional_kwargs
                print(f"'agent': '{content}'")
            elif key == "tools":
                for m in chunk["tools"]["messages"]:
                    print(f"'tools': '{m.content}'")
            results.append(chunk)
        return results
    except Exception as e:
        if "Recursion limit" in str(e):
            print(f"[재귀 한도 도달] {e}")
        else:
            print(f"Error processing stream: {e}")
        return results

## 실행: AI 최신 논문 1편 검색 → 모델 허브 검색 → Notion 정리 (매일 1회)

1. **Hugging Face MCP**로 AI 관련 최신 논문 1편만 검색 (paper_search, results_limit=1)

2. **Hugging Face MCP**로 논문 관련 모델/데이터셋 검색  
   hub_repo_search(query=논문 키워드, repo_types=["model", "dataset"]), 필요 시 hub_repo_details로 URL·상세 정보 수집

3. **Hugging Face MCP** hub_repo_search로 관련 모델 추가 검색 후 정보 수집

4. **정리**  
   MCP 도구 결과를 그대로 쓰고, LLM은 표(테이블)·제목·[링크](URL) 포맷 적용. 논문·모델 설명은 최소 5줄 이상, 쉬운 말로 작성.

5. **Notion MCP** create-pages로 작성. content에 본문을 마크다운 문자열 하나로 넣어 링크가 클릭 가능하게 (API-post-page 사용 금지).

6. **매일 1회 실행** 시 당일 논문 1편이 날짜별 페이지(제목: AI 최신 논문 및 모델 정리 (YYYY-MM-DD))로 추가됨.

### 매일 1회 자동 실행 설정 (선택)

- **macOS/Linux (cron)**  
  `crontab -e` 후 예시 — 매일 오전 9시 실행  
  `0 9 * * * cd /path/to/inflearn-langgraph-lecture2 && .venv/bin/jupyter nbconvert --to notebook --execute --inplace 19.mcp_huggingface_notion.ipynb`

- **GitHub Actions**  
  워크플로에서 `schedule: cron('0 0 * * *')` (매일 0시 UTC) 등으로 동일 nbconvert 명령 실행

In [5]:
from datetime import datetime
from langchain_core.messages import HumanMessage

notion_parent_id = os.getenv("NOTION_PARENT_PAGE_ID", "").strip()
run_date = datetime.now().strftime("%Y-%m-%d")  # 매일 1회 실행 시 날짜별 페이지 제목에 사용

query = f"""
다음 순서로 진행해줘. **MCP를 최대한 사용**하고, LLM은 4) 구조·포맷만 보조로 써줘. **오늘 AI 최신 논문 1편만** 대상.

1) [Hugging Face MCP] 연구 논문(papers) 검색: **AI 관련 최신 논문**을 검색해줘. paper_search 사용 시 **query는 최소 3글자 이상**이어야 함 (예: "AI 최신 논문", "recent AI paper", "machine learning"). **results_limit=1** 로 설정해줘. 논문 제목, 링크, 요약을 수집해줘.

2) [Hugging Face MCP] **반드시 Hugging Face MCP 도구**로 논문 관련 모델/데이터셋 링크 찾기: **hub_repo_search**를 사용해줘. 검색된 논문 제목이나 키워드를 query에 넣고, **repo_types에 "model"과 "dataset"**을 포함해서 호출해줘. (예: hub_repo_search(query=논문제목 또는 핵심키워드, repo_types=["model", "dataset"], limit=5)). 결과로 나온 각 리포지토리의 URL·이름·설명을 수집해줘. 필요하면 **hub_repo_details**로 리포지토리 상세 정보(URL 등)를 추가로 가져와도 됨. 이 단계에서 논문에서 찾은 모델/데이터셋 링크는 **전부 Hub 도구 결과**로 채워줘.

3) [Hugging Face MCP] 모델 허브 검색: **hub_repo_search**로 해당 논문·주제와 관련된 AI 모델을 한 번 더 검색하고(예: repo_types=["model"], sort="downloads" 또는 "trendingScore"), 모델 정보(설명, 다운로드 수, 링크 등)를 가져와줘.

4) 정리(LLM 보조): 1~3)에서 MCP로 수집한 내용을 바탕으로 **본문 구조와 링크 포맷**을 잡아줘. **설명은 한 줄 요약이 아니라 최소 5줄 이상**으로 쓰고, **쉬운 말**(비전공자도 이해 가능)로 풀어서 써줘. MCP 결과를 그대로 인용해도 되고, 같은 내용을 더 풀어서 써도 됨.
   - **가독성**: 상단에 짧은 소개(오늘 수집한 AI 논문 1편과 관련 모델/데이터셋입니다). 논문·모델은 **표(테이블)** 로 정리해줘.
     - **설명은 한 줄 요약 금지.** 논문·모델 모두 **설명을 최소 5줄 이상** 쓰고, **쉬운 말**로 풀어서 써줘 (비전공자도 이해할 수 있게).
     - 논문: | 제목 | 설명 (최소 5줄, 쉬운 말로) | 링크 | → 링크 열은 반드시 [논문 보기](URL) 형태.
     - 모델: | 모델/리포지토리명 | 설명 (최소 5줄, 쉬운 말로) | 링크 | → 링크 열은 [이름](URL) 형태.
   - **모든 URL은 [표시텍스트](URL) 마크다운 링크**로만 넣어줘. MCP 결과에 나온 URL을 그대로 사용.

5) **Notion API 호출은 하지 마라.** 4)에서 정리한 본문(표·링크 포함)을 **그대로 최종 응답으로만 출력**해줘.
"""

stream_generator = agent.astream(
    {"messages": [HumanMessage(content=query)]},
    config={"recursion_limit": 20},
)
all_chunks = await process_stream(stream_generator)

# 마지막 AI 메시지에서 정리된 마크다운 추출
formatted_markdown = None
if all_chunks:
    for chunk in reversed(all_chunks):
        if "agent" in chunk:
            msgs = chunk["agent"].get("messages", [])
            if msgs and hasattr(msgs[0], "content") and msgs[0].content:
                formatted_markdown = msgs[0].content
                break
    if formatted_markdown:
        print("\n[정리된 본문 추출 완료] Notion 페이지 생성 시도...")
    if all_chunks:
        print(all_chunks[-1])

# Notion 페이지 생성: 사용 가능한 도구로 부모 페이지 아래에 새 페이지 생성
if formatted_markdown and notion_parent_id:
    # 1) create-pages / notion-create-pages 계열 도구 먼저 탐색
    create_tool = next((t for t in tool_list if "create" in t.name.lower() and "page" in t.name.lower() and "API" not in t.name), None)
    # 2) 없으면 정확한 이름 시도 (Notion MCP가 다른 이름을 쓸 수 있음)
    if not create_tool:
        for name in ("notion-create-pages", "notion_create_pages", "create-pages", "create_pages"):
            create_tool = next((t for t in tool_list if t.name == name or t.name.endswith(name)), None)
            if create_tool:
                break
    if create_tool:
        try:
            result = await create_tool.ainvoke({
                "parent": {"page_id": notion_parent_id},
                "pages": [{
                    "properties": {"title": f"AI 최신 논문 및 모델 정리 ({run_date})"},
                    "content": formatted_markdown,
                }],
            })
            print("Notion 페이지 생성 결과:", result)
        except Exception as e:
            print("Notion create-pages 호출 오류:", e)
    else:
        # 3) API-post-page로 블록 배열 전달 (마크다운 → paragraph/heading 블록으로 변환)
        import re
        def _md_to_notion_blocks(md: str):
            blocks = []
            for line in md.strip().split("\n"):
                line = line.rstrip()
                if not line:
                    continue
                # [텍스트](URL) → rich_text에 링크로 넣기
                rich = []
                pos = 0
                for m in re.finditer(r"\[([^\]]+)\]\(([^)]+)\)", line):
                    if m.start() > pos:
                        rich.append({"type": "text", "text": {"content": line[pos:m.start()]}})
                    rich.append({"type": "text", "text": {"content": m.group(1), "link": {"url": m.group(2)}}})
                    pos = m.end()
                if pos < len(line):
                    rich.append({"type": "text", "text": {"content": line[pos:]}})
                if not rich:
                    rich = [{"type": "text", "text": {"content": line}}]
                if line.startswith("### "):
                    blocks.append({"type": "heading_3", "heading_3": {"rich_text": rich}})
                elif line.startswith("## "):
                    blocks.append({"type": "heading_2", "heading_2": {"rich_text": rich}})
                elif line.startswith("|") and "|" in line[1:]:
                    blocks.append({"type": "paragraph", "paragraph": {"rich_text": rich}})
                else:
                    blocks.append({"type": "paragraph", "paragraph": {"rich_text": rich}})
            return blocks if blocks else [{"type": "paragraph", "paragraph": {"rich_text": [{"type": "text", "text": {"content": md[:2000]}}]}}]
        api_tool = next((t for t in tool_list if t.name == "API-post-page"), None)
        if api_tool:
            try:
                children = _md_to_notion_blocks(formatted_markdown)
                result = await api_tool.ainvoke({
                    "parent": {"page_id": notion_parent_id},
                    "properties": {"title": {"title": [{"type": "text", "text": {"content": f"AI 최신 논문 및 모델 정리 ({run_date})"}}]}},
                    "children": children,
                })
                print("Notion 페이지 생성 결과 (API-post-page):", result)
            except Exception as e:
                print("API-post-page 호출 오류:", e)
                print("사용 가능한 전체 툴 이름:", [t.name for t in tool_list])
        else:
            print("사용 가능한 전체 툴 이름:", [t.name for t in tool_list])
            print("[안내] Notion 페이지 생성 도구를 찾을 수 없습니다. 정리된 본문을 Notion에 수동으로 붙여넣거나, .env의 NOTION_PARENT_PAGE_ID를 확인하세요.")
elif formatted_markdown and not notion_parent_id:
    print("[안내] .env에 NOTION_PARENT_PAGE_ID를 설정하면 여기서 자동으로 Notion 페이지가 생성됩니다.")


'agent': '{'tool_calls': [{'id': 'call_sxRRHKsUfr77kUfwJTadFGOD', 'function': {'arguments': '{"query":"recent AI paper","results_limit":1}', 'name': 'paper_search'}, 'type': 'function'}], 'refusal': None}'
'tools': '120 papers matched the query 'recent AI paper'. Here are the first 1 results.

---

## Generative AI for Film Creation: A Survey of Recent Advances

Published on Published on 11 Apr, 2025
**Authors:** Ruihan Zhang, Borou Yu, Jiajian Min, Yetong Xin, Zheng Wei, Juncheng Nemo Shi, Mingzhen Huang, Xianghao Kong ([refkxh](https://hf.co/refkxh)), and 17 more.

### Abstract

Generative AI (GenAI) is transforming filmmaking, equipping artists with tools like text-to-image and image-to-video diffusion, neural radiance fields, avatar generation, and 3D synthesis. This paper examines the adoption of these technologies in filmmaking, analyzing workflows from recent AI-driven films to understand how GenAI contributes to character creation, aesthetic styling, and narration. We explore k